In [1]:
!wget http://cluster.ischool.drexel.edu/~jz85/SDSSLogViewer/sampledata/median.csv

--2026-04-23 08:50:41--  http://cluster.ischool.drexel.edu/~jz85/SDSSLogViewer/sampledata/median.csv
Resolving cluster.ischool.drexel.edu (cluster.ischool.drexel.edu)... 129.25.203.170
Connecting to cluster.ischool.drexel.edu (cluster.ischool.drexel.edu)|129.25.203.170|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 302235399 (288M) [text/plain]
Saving to: ‘median.csv’

median.csv            9%[>                   ]  28.33M   272KB/s    eta 15m 48s^C


In [ ]:
# 1. Clone the repository structure without downloading all files (blobs)
!git clone --depth 1 --filter=blob:none --sparse https://github.com/SQL-Storm/SQLStorm.git

# 2. Navigate into the repository
%cd SQLStorm

# 3. Configure Git to only download the specific subdirectory you need
!git sparse-checkout set v1.0/stackoverflow/queries

Cloning into 'SQLStorm'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 54 (delta 3), reused 46 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 12.63 MiB | 25.50 MiB/s, done.
Resolving deltas: 100% (3/3), done.
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 7 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), 5.31 KiB | 5.31 MiB/s, done.
/kaggle/working/SQLStorm
remote: Enumerating objects: 17314, done.
remote: Counting objects: 100% (17314/17314), done.
remote: Compressing objects: 100% (16077/16077), done.
remote: Total 17314 (delta 1235), reused 17308 (delta 1233), pack-reused 0 (from 0)
Receiving objects: 100% (17314/17314), 7.75 MiB | 15.87 MiB/s, done.
Resolving deltas: 100% (1235/1235), done.
Updating files: 100% (18258/18258), don

In [2]:
path="/kaggle/working/SQLStorm/v1.0/stackoverflow/queries"
import pandas as pd
import os


def load_sql_dir(directory):
    """
    Load every .sql file in `directory` into a pandas DataFrame.

    Parameters
    ----------
    directory : str
        Path to the folder containing .sql files.

    Returns
    -------
    pd.DataFrame
        Columns: ['file_name', 'sql_text']
    """
    records = []

    for file_name in sorted(os.listdir(directory)):
        if file_name.lower().endswith(".sql"):
            file_path = os.path.join(directory, file_name)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    sql_text = f.read()
                records.append({"file_name": file_name, "sql_text": sql_text})
            except (FileNotFoundError, PermissionError, UnicodeDecodeError) as e:
                print(f"[SKIP] {file_path} → {e}")

    df = pd.DataFrame(records, columns=["file_name", "sql_text"])
    print(f"Loaded {len(df)} .sql files from {directory}")
    return df
df_stackoverflow=load_sql_dir(path)

Loaded 18251 .sql files from /kaggle/working/SQLStorm/v1.0/stackoverflow/queries


In [3]:
df_stackoverflow.columns

Index(['file_name', 'sql_text'], dtype='object')

In [ ]:
import torch
import pandas as pd
import sqlparse
import re
import numpy as np
from transformers import BertTokenizer, BertModel
from torch.utils.data import DataLoader, SequentialSampler
from tqdm.auto import tqdm

class QueryEmbedder:
    def __init__(self, model_path="/kaggle/input/models/nileshthota/bert-finetuned-for-sql/pytorch/default/1", batch_size=32, max_length=128):
        """
        Initializes the embedder, loading the fine-tuned BERT model and tokenizer.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Initializing QueryEmbedder on {self.device}...")
        
        # Load tokenizer and base model (BertModel is used to extract hidden states)
        self.tokenizer = BertTokenizer.from_pretrained(model_path)
        self.model = BertModel.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()
        
        self.batch_size = batch_size
        self.max_length = max_length
        self.sneaky_number_pattern = re.compile(r'^[-+]?(?:\d+\.\d*|\d*\.\d+|\d+)(?:[eE][-+]?\d+)?$')

    def _normalize_sql(self, sql):
        """Lowercase and collapse whitespace."""
        if not isinstance(sql, str):
            return ""
        sql = sql.lower().strip()
        return re.sub(r'\s+', ' ', sql)

    def _mask_literals(self, sql):
        """Robustly tokenize SQL and mask literals to match training data format."""
        sql = self._normalize_sql(sql)
        if not sql:
            return ""
            
        parsed = sqlparse.parse(sql)
        if not parsed:
            return ""
            
        tokens = []
        from sqlparse.tokens import Number, String
        
        for token in parsed[0].flatten():
            if token.is_whitespace:
                continue
                
            val = token.value
            if token.ttype in Number or self.sneaky_number_pattern.fullmatch(val) or re.fullmatch(r'^[-+]?\d+$', val):
                tokens.append('NUM_LITERAL')
            elif token.ttype in String:
                tokens.append('STR_LITERAL')
            elif val.startswith('0x') and len(val) > 2:
                tokens.append('HEX_LITERAL')
            else:
                tokens.append(val)
                
        return " ".join(tokens)

    def generate_embeddings(self, df, text_column='sql_text'):
        """
        Takes a pandas DataFrame and returns a numpy array of BERT embeddings.
        """
        if text_column not in df.columns:
            raise ValueError(f"Column '{text_column}' not found in DataFrame.")

        print("Preprocessing and masking literals...")
        # Apply tokenization and masking to the entire column
        processed_queries = df[text_column].apply(self._mask_literals).tolist()

        print("Tokenizing for BERT...")
        # Tokenize using the BERT tokenizer
        inputs = self.tokenizer(
            processed_queries,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Use PyTorch DataLoader to handle memory-efficient batching
        dataset = torch.utils.data.TensorDataset(inputs['input_ids'], inputs['attention_mask'])
        sampler = SequentialSampler(dataset)
        dataloader = DataLoader(dataset, sampler=sampler, batch_size=self.batch_size)

        all_embeddings = []

        print("Generating embeddings...")
        with torch.no_grad(): # Crucial for preventing memory leaks during inference
            for batch in tqdm(dataloader, desc="Inference Batches"):
                input_ids = batch[0].to(self.device)
                attention_mask = batch[1].to(self.device)

                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                
                # Extract the [CLS] token representation (index 0) from the last hidden state
                cls_embeddings = outputs.last_hidden_state[:, 0, :]
                all_embeddings.append(cls_embeddings.cpu().numpy())

        # Concatenate all batches into a single 2D numpy array
        return np.vstack(all_embeddings)

# --- Usage Example ---
embedder = QueryEmbedder( batch_size=64)
embeddings_array = embedder.generate_embeddings(df_stackoverflow, text_column='sql_text')
np.save("bert_embeds.npy",embeddings_array)

Initializing QueryEmbedder on cuda...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/nileshthota/bert-finetuned-for-sql/pytorch/default/1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Preprocessing and masking literals...
Tokenizing for BERT...
Generating embeddings...


Inference Batches:   0%|          | 0/286 [00:00<?, ?it/s]

In [6]:
embeddings_array=np.load("bert_embeds.npy")

In [5]:
X_bert=np.array(embeddings_array)

In [6]:
import random
y=np.random.normal(loc=5,scale=np.sqrt(10),size=len(X_bert))

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_val,y_train,y_val=train_test_split(X_bert,y,random_state=42)

In [8]:
import numpy as np
import pandas as pd
import time
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import LinearSVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Optional – install if needed: pip install xgboost catboost
try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed. Skipping.")
try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("CatBoost not installed. Skipping.")

# Models – all chosen for speed on 18k samples
models = {
    # 'Linear Regression': LinearRegression(),
    # 'SGD Regressor': SGDRegressor(max_iter=1000, tol=1e-3, random_state=42), 
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    # 'Linear SVR': LinearSVR(max_iter=10000, random_state=42),   
}


results = []

for name, regressor in models.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', regressor)
    ])
    
    start = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start
    
    y_train_pred = pipeline.predict(X_train)
    y_val_pred = pipeline.predict(X_val)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    val_r2 = r2_score(y_val, y_val_pred)
    
    results.append({
        'Model': name,
        'Train Time (s)': round(train_time, 2),
        'Train RMSE': train_rmse,
        'Val RMSE': val_rmse,
        'Train MAE': train_mae,
        'Val MAE': val_mae,
        'Train R2': train_r2,
        'Val R2': val_r2
    })
    print(f"{name} finished in {train_time:.2f}s")

results_df = pd.DataFrame(results)
print("\nResults:")
print(results_df.to_string(index=False))

Decision Tree finished in 15.14s

Results:
        Model  Train Time (s)  Train RMSE  Val RMSE  Train MAE  Val MAE  Train R2    Val R2
Decision Tree           15.14     3.04876  3.221645   2.420382 2.554779  0.051985 -0.052107


In [ ]:
def get_results(X_train,y_train,X_val,y_val):
    import numpy as np
    import pandas as pd
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LinearRegression
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.svm import SVR
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    # Optional: handle XGBoost and CatBoost if installed
    try:
        from xgboost import XGBRegressor
        XGB_AVAILABLE = True
    except ImportError:
        XGB_AVAILABLE = False
        print("XGBoost not installed. Skipping XGBoost.")

    try:
        from catboost import CatBoostRegressor
        CATBOOST_AVAILABLE = True
    except ImportError:
        CATBOOST_AVAILABLE = False
        print("CatBoost not installed. Skipping CatBoost.")

    models = {
        'Linear Regression': LinearRegression(),
        'Decision Tree': DecisionTreeRegressor(random_state=42),
        'Random Forest': RandomForestRegressor(random_state=42),
        'SVR': SVR(),
    }

    if XGB_AVAILABLE:
        models['XGBoost'] = XGBRegressor(random_state=42)
    if CATBOOST_AVAILABLE:
        models['CatBoost'] = CatBoostRegressor(silent=True, random_state=42)

    results = []

    for name, regressor in models.items():
        # Create pipeline: StandardScaler + regressor
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', regressor)
        ])
        
        pipeline.fit(X_train, y_train)
        
        y_train_pred = pipeline.predict(X_train)
        y_val_pred = pipeline.predict(X_val)
        
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
        val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
        train_mae = mean_absolute_error(y_train, y_train_pred)
        val_mae = mean_absolute_error(y_val, y_val_pred)
        train_r2 = r2_score(y_train, y_train_pred)
        val_r2 = r2_score(y_val, y_val_pred)
        
        results.append({
            'Model': name,
            'Train RMSE': train_rmse,
            'Val RMSE': val_rmse,
            'Train MAE': train_mae,
            'Val MAE': val_mae,
            'Train R2': train_r2,
            'Val R2': val_r2
        })

    results_df = pd.DataFrame(results)
    # print(results_df.to_string(index=False))
    return results_df